# 02b. Классификация замен мтДНК

## Постановка

Дополнительный набор 7 Lake 2024 содержит MLC-score для всех 49 704 возможных
однонуклеотидных замен митохондриального генома. Задача — классифицировать эту
совокупность по степени тяжести, используя MLC-score вместе с независимыми
источниками свидетельств.

Решаются три вопроса:

1. Сколько упорядоченных классов тяжести разделимо в признаковом пространстве.
2. Какая модель обеспечивает классификацию.
3. Какое подмножество признаков для этого необходимо.

Одноклассовый Isolation Forest из ноутбука 02 сохраняется как **базовая линия**:
он переобучается на тех же признаках и оценивается на том же протоколе, что и
супервизные модели. Вычислительная часть — в
[`scripts/features.py`](../scripts/features.py) и
[`scripts/classification.py`](../scripts/classification.py).

## Признаковая панель

Шесть содержательных предикторов, десять колонок после кодирования категорий.

| Признак | Источник | Роль |
|---|---|---|
| `mlc_score` | Lake 2024, набор 7 | Локальное ограничение отбора, для конкретной замены |
| `phyloP100way` | UCSC, 100 позвоночных | Межвидовая консервативность позиции |
| `hom_rarity_soft` | gnomAD и HelixMT | Один частотный сигнал: редкость в гомоплазмическом состоянии |
| `cons_*` | Аннотация VEP | Класс последствия: missense, synonymous, truncating, некодирующая РНК, межгенный |
| `codon_pos2_any`, `codon_pos3_any` | Рамка считывания гена | Положение в кодоне |

Панель заменила прежнюю из девяти признаков, куда входили четыре отдельные меры
частоты. Их попарная корреляция Спирмена достигала 0.99, то есть они несли около
полутора независимых размерностей, и полная панель работала **хуже**, чем её
подмножество из пяти признаков. Класс последствия теперь входит напрямую, а не
через положение в кодоне, которое было лишь его приближением.

## Два структурных ограничения

**Оба критерия разметки частично восстановимы из признаков.** Нейтральный набор 8
собран из гаплогруппных вариантов, частых по построению, и из нижнего дециля
phyloP. Поэтому частотный признак кодирует первый критерий, а phyloP — второй.
На когорте из одних гаплогруппных нейтральных `hom_rarity_soft` в одиночку
разделяет benign и pathogenic с AUC 0.98 — величина, отражающая правило отбора,
а не биологию. Ограничение оценки этой когортой снимает phyloP-цикличность, но
концентрирует частотную, поэтому чистой когорты нет ни одной и приводятся обе.

**Различение аллелей ограничено.** Только `mlc_score` и `hom_rarity_soft`
различают альтернативные аллели одной позиции, остальные признаки позиционны.
Примерно у шести из десяти неразмеченных позиций все три замены получают
идентичный вектор признаков, поэтому модель ранжирует позиции, а не аллели.

Неразмеченные варианты в обучении не участвуют; предсказания ниже порога
уверенности помечаются как `uncertain`.

In [ ]:
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

import features as ft
import classification as cl

RESULT_DIR = PROJECT_ROOT / "results/classification"
FIGURE_DIR = PROJECT_ROOT / "results/figures/classification"
FEATURE_TABLE = PROJECT_ROOT / "data/processed/model_features.tsv"

# Recompute from scratch when True. The thorough budget takes about an hour on
# 16 cores and reproduces the tables already stored in RESULT_DIR.
RUN_COMPUTATION = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

def show(name: str) -> None:
    display(Image(filename=str(FIGURE_DIR / name)))

def table(name: str, **kwargs) -> pd.DataFrame:
    return pd.read_csv(RESULT_DIR / name, sep="\t", **kwargs)

print("признаков в панели:", len(ft.FEATURE_COLUMNS))
print(ft.FEATURE_COLUMNS)

## 1. Сборка признаков и классов

Таблица признаков строится один раз и сохраняется, чтобы ноутбук и скрипт не
расходились. Основа — мастер-таблица из ноутбука 00, к ней присоединяется
phyloP по координате, после чего выводятся признаки панели и три схемы меток.

Метки формируются из наборов 3, 8 и 9 по правилу приоритета

```text
набор 9 (курированные патогенные) > набор 3 (клиническая группа) > набор 8 (нейтральные)
```

Набор 3 приоритетнее набора 8, поскольку содержит индивидуальную клиническую
оценку варианта, тогда как принадлежность к набору 8 задана групповым правилом.

In [ ]:
if not FEATURE_TABLE.exists():
    raw_dir = PROJECT_ROOT / "data/raw/lake_2024_supplement_data"
    built = ft.build_features(
        PROJECT_ROOT / "data/processed/master_snv_table.tsv",
        PROJECT_ROOT / "data/raw/phyloP100way.tsv",
        raw_dir / "supplementary_dataset_3.tsv",
        raw_dir / "supplementary_dataset_8.tsv",
        raw_dir / "supplementary_dataset_9.tsv",
    )
    FEATURE_TABLE.parent.mkdir(parents=True, exist_ok=True)
    built.to_csv(FEATURE_TABLE, sep=chr(9), index=False)

data = pd.read_csv(FEATURE_TABLE, sep="\t", low_memory=False)
print(f"строк: {len(data):,}")
display(data["consequence_class"].value_counts().rename_axis("класс последствия")
        .to_frame("замен"))
display(table("label_scheme_audit.tsv"))

In [ ]:
show("label_scheme_inventory.png")
show("forced_features_by_class.png")

## 2. Протокол оценки

**Группировка по позиции.** Разбиение выполняет `StratifiedGroupKFold` с
группировкой по координате мтДНК. Замены одной позиции разделяют значения
позиционных признаков, и без группировки они попадают одновременно в обучение и
контроль, завышая оценку.

**Повторность.** Пять блоков в пяти повторах; дисперсия между повторами служит
мерой устойчивости.

**Метрика.** Основная — macro F1: при соотношении классов 8 217 : 116 : 91 доля
правильных ответов неинформативна, поскольку тривиальная модель достигает 97%.
Дополнительно приводится квадратично взвешенная каппа, учитывающая
упорядоченность классов.

Сравниваются восемь моделей, включая порядковый лес с декомпозицией
Франка — Холла и стратифицированную базовую линию.

In [ ]:
if RUN_COMPUTATION:
    args = argparse.Namespace(
        input=FEATURE_TABLE,
        dataset3=PROJECT_ROOT / "data/raw/lake_2024_supplement_data/supplementary_dataset_3.tsv",
        dataset8=PROJECT_ROOT / "data/raw/lake_2024_supplement_data/supplementary_dataset_8.tsv",
        dataset9=PROJECT_ROOT / "data/raw/lake_2024_supplement_data/supplementary_dataset_9.tsv",
        output_dir=RESULT_DIR, figure_dir=FIGURE_DIR,
        budget="thorough", folds=5, repeats=5, tune_folds=5, selection_repeats=3,
        bootstrap=2000, finalists=3, min_coverage=0.70, target_fpr=0.05,
        viability_threshold=0.10, scheme_tolerance=0.05,
        primary_metric="macro_f1", scoring="f1_macro",
        random_state=20260901, n_jobs=-1, skip_nested=False,
    )
    results = cl.run(args)

manifest = json.loads((RESULT_DIR / "selection_manifest.json").read_text(encoding="utf-8"))
print("Модель:            ", manifest["selected_model"])
print("Схема меток:       ", manifest["selected_scheme"], manifest["selected_classes"])
print("Признаки:          ", manifest["selected_features"])
print("Порог уверенности: ", round(manifest["confidence_threshold"]["threshold"], 3))

## 3. Сравнение моделей

Распределения macro F1 по всем блокам всех повторов, отдельно для каждой схемы.
Существенна не только медиана, но и разброс: при сопоставимых медианах
предпочтительна модель с меньшей дисперсией между разбиениями.

In [ ]:
show("model_comparison.png")
table("model_scheme_summary.tsv")[
    ["scheme", "model", "macro_f1_mean", "macro_f1_sem",
     "balanced_accuracy_mean", "quadratic_kappa_mean", "roc_auc_mean"]
].round(4)

Прямое сравнение средних некорректно: блоки общие для всех моделей, наблюдения
парные, а межблочная дисперсия сопоставима с различиями между моделями.
Применяется ранжирование внутри каждого блока, критерий Фридмана и апостериорный
тест Немени с критической разностью. Модели, чьи средние ранги различаются менее
чем на эту величину, статистически неразличимы.

In [ ]:
for scheme in ["2class", "3class", "4class"]:
    show(f"critical_difference_{scheme}.png")
table("model_ranking_nemenyi.tsv").round(4)

## 4. Сколько классов оправдано

Значения macro F1 для схем с разным числом классов несопоставимы. Детализация
оценивается двумя независимыми критериями.

**Цена детализации.** Предсказания каждой схемы сворачиваются до различения
benign и pathogenic на вариантах, размеченных во всех трёх схемах. Ранжирующая
статистика — вероятность самого тяжёлого класса; она безразмерна, поэтому
сравнима между схемами.

**Разделимость классов.** Каждый класс оценивается по MCC «один против
остальных». Сравнение полноты со стратифицированной базовой линией здесь
непригодно: у неё полнота класса равна его доле, то есть 0.97 для `benign` и
0.014 для `vus`. MCC «один против остальных» равен нулю при случайном
предсказании независимо от доли класса.

In [ ]:
show("label_scheme_tradeoff.png")
display(table("label_scheme_comparison.tsv").round(4))
display(table("label_scheme_per_class.tsv").round(4))
print("Правило отбора:", manifest["scheme_choice"]["reason"])

In [ ]:
show("confusion_matrix_by_scheme.png")

## 5. Отбор признаков

Жадный прямой отбор: `mlc_score` и `phyloP100way` включены исходно, далее
добавляется признак с наибольшим приростом macro F1. Размер подмножества
определяется правилом одной стандартной ошибки — наименьшее подмножество,
отличающееся от наилучшего не более чем на одну стандартную ошибку.

Порядок добавления не эквивалентен перестановочной важности: первый отражает
прирост при добавлении, вторая — потерю при удалении, и при коррелированных
признаках эти величины расходятся.

In [ ]:
show("feature_selection_curve.png")
table("feature_selection_curve.tsv").round(4)

## 6. Характеристика выбранной модели

In [ ]:
show("selected_model_confusion_matrix.png")
display(table("selected_model_confusion_matrix.tsv", index_col=0))
show("selected_model_per_class_performance.png")
table("selected_model_per_class_metrics.tsv").round(4)

Калибровка проверяет соответствие предсказанных вероятностей наблюдаемым
частотам. Это существенно, поскольку порог отнесения к `uncertain` задаётся
непосредственно на этих величинах.

Перестановочная важность измерена на отложенных блоках. Доверительные интервалы
получены кластерным бутстрепом по позициям мтДНК: ресемплируются позиции
целиком, поскольку варианты внутри позиции зависимы. Вложенная кросс-валидация
служит проверкой на смещение отбора.

In [ ]:
show("selected_model_calibration.png")
show("selected_model_roc_pr.png")
show("selected_model_permutation_importance.png")
display(table("selected_model_bootstrap_ci.tsv").round(4))
nested = RESULT_DIR / "nested_cv_finalists.tsv"
if nested.exists():
    display(pd.read_csv(nested, sep="\t")
            .groupby("model")[["macro_f1", "balanced_accuracy", "quadratic_kappa"]]
            .agg(["mean", "std"]).round(4))

## 7. Isolation Forest как базовая линия

Одноклассовый лес переобучается на выбранных признаках и оценивается на тех же
позиционно-сгруппированных блоках, что и супервизные модели. Внутри каждого
блока он обучается на части нейтральных обучающих строк, а порог калибруется на
остальных — это воспроизводит разделение нейтрального набора на обучающую и
валидационную части и правило T95 из ноутбука 02.

Сравнение проводится на подзадаче benign против pathogenic, поскольку это
единственный вопрос, который можно задать одноклассовой модели: патогенных
меток она не видит.

Ключевой элемент сравнения — **одиночные признаки на тех же блоках и по тому же
правилу калибровки**. Если многопризнаковый лес не превосходит один признак, то
он этот признак и воспроизводит. Ориентация каждого признака берётся из
обучающих строк, никогда из контрольных.

In [ ]:
show("isolation_forest_baseline.png")
table("isolation_forest_baseline_summary.tsv")[
    ["method", "kind", "roc_auc_mean", "average_precision_mean",
     "recall_at_target_fpr_mean", "realised_fpr_mean"]
].round(4)

Интерпретация. Одноклассовая постановка не требует патогенных меток, что важно
при их числе в 91 вариант, но её нейтральный эталон неоднороден: половина —
частые гаплогруппные варианты, половина — никогда не наблюдавшиеся позиции
нижнего дециля phyloP. Две несовместимые популяции, объединённые в один класс
«нормы», дают широкую область нормальности, и потому доля вариантов вне
нейтрального домена оказывается большой.

## 8. Контроль циклической связи phyloP с разметкой

Признак не исключается — пересчитывается оценка на когорте, где остались только
гаплогруппные нейтральные варианты. Диагностически значимо не снижение метрики,
которое ожидаемо, а изменение порядка моделей.

Границы этой проверки следует оговорить: удаляются циклически отобранные
нейтральные варианты, а не сам признак phyloP. При этом когорта становится
однородно гаплогруппной, то есть частотная цикличность в ней усиливается.
Ни одна из двух когорт не свободна от связи признаков с определением метки.

In [ ]:
show("leakage_control.png")
table("leakage_control_comparison.tsv").round(4)

## 9. Применение ко всем возможным заменам

Модель обучается на полной размеченной когорте и применяется ко всем 49 704
заменам. Порог уверенности выбран как значение, максимизирующее macro F1 на
отложенных данных при покрытии не менее 70%.

In [ ]:
show("abstention_tradeoff.png")
predictions = table("all_substitution_predictions.tsv")
display(table("unlabeled_call_counts.tsv").round(4))
print(f"Всего замен: {len(predictions):,}")
print(f"Неразмеченных: {(~predictions['is_training_variant']).sum():,}")

Существенное ограничение интерпретации: высокая доля класса `pathogenic` среди
неразмеченных вариантов не означает соответствующей доли клинически патогенных
замен. Признак `hom_rarity_soft` принимает максимальное значение для всякой
замены, которую никто не наблюдал, а таких семь из десяти.

In [ ]:
show("prediction_class_inventory.png")
show("prediction_score_distributions.png")
show("genome_prediction_map.png")
show("feature_space_projection.png")
show("decision_landscape.png")

In [ ]:
agreement = RESULT_DIR / "agreement_with_isolation_forest.tsv"
if agreement.exists():
    display(pd.read_csv(agreement, sep="\t", index_col=0))

top = (predictions[~predictions["is_training_variant"]]
       .sort_values("expected_severity", ascending=False).head(25))
top[["variant_id", "position", "predicted_class", "called_class", "confidence",
     "expected_severity", "mlc_score", "phyloP100way"]].round(4)

## 10. Результаты и ограничения

### Установленные результаты

Протокол определяет модель, статистически не уступающую остальным; число
упорядоченных классов, подтверждаемое разделимостью в признаковом пространстве;
достаточное подмножество признаков по правилу одной стандартной ошибки; величину
превосходства над одноклассовой базовой линией и над одиночными признаками.

### Ограничения интерпретации

Предсказанный класс `pathogenic` не эквивалентен клинической патогенности: он
выражает сходство с курированным набором из 91 варианта по используемым
признакам, часть которых отражает популяционную частоту.

Оба критерия отбора нейтрального набора частично восстановимы из признаков,
поэтому разделение benign и pathogenic отчасти воспроизводит правило разметки.
Ни одна из контрольных когорт не устраняет обе цикличности одновременно.

Модель ранжирует позиции, а не аллели: у шести из десяти неразмеченных позиций
все три замены получают идентичный вектор признаков.

Промежуточные классы оценены на выборках порядка десятков вариантов; их метрики
нестабильны независимо от формальной ширины интервалов.

Вся разметка происходит из одной публикации и наследует её критерии отбора.

### Применимость

Предсказания пригодны как ось приоритизации для последующей проверки. Столбец
`expected_severity` применим для ранжирования, `called_class` — для формирования
когорт, а варианты, отнесённые к `uncertain`, отражают недостаточность
признакового пространства для соответствующего решения.